# Data Indgestion 

In [1]:
# Document Structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="this is some kind of a text which is in the file", 
    metadata={
        "source" : "smth resource", 
        "pages" : 1, 
        "auther" : "rivindu", 
        "data_created" : "2023-10-01"

    }
)

In [3]:
doc

Document(metadata={'source': 'smth resource', 'pages': 1, 'auther': 'rivindu', 'data_created': '2023-10-01'}, page_content='this is some kind of a text which is in the file')

In [4]:
from langchain_classic.document_loaders import TextLoader, DirectoryLoader

loader = TextLoader("../data/TextFiles/sample.txt", encoding="utf-8")

d:\RAG-Cookbook\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
document = loader.load()

In [6]:
directory_loader = DirectoryLoader(
    path="../data/TextFiles",
    glob="*.txt",
    loader_cls=TextLoader
)

directory_loader.load()

[Document(metadata={'source': '..\\data\\TextFiles\\sample.txt'}, page_content='Python Programming Language\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum in 1991, Python emphasizes code clarity through its use of whitespace and straightforward syntax.\n\nKey Features:\n- Easy to learn with beginner-friendly syntax\n- Dynamically typed, allowing flexible variable declaration\n- Extensive standard library for various tasks\n- Supports multiple programming paradigms: procedural, object-oriented, and functional\n- Cross-platform compatibility (Windows, macOS, Linux)\n- Strong community support and rich ecosystem of third-party packages\n\nCommon Applications:\n- Web development with frameworks like Django and Flask\n- Data science and machine learning using NumPy, Pandas, and TensorFlow\n- Automation and scripting\n- Artificial intelligence and natural language processing\n- Scientific computing\n- Game dev

In [7]:
from langchain_classic.document_loaders import PyPDFLoader, DirectoryLoader
pdf_doc = DirectoryLoader("../data/pdfFiles/", 
                          loader_cls=PyPDFLoader, 
                          glob="*.pdf")
pdf_doc.load()


[Document(metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-30T11:21:16+00:00', 'author': 'Rivindu Ashinsa', 'title': "Rivindu Ashinsa's CV", 'source': '..\\data\\pdfFiles\\RivinduAshinsaResumeGenAI.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Rivindu Ashinsa\nComputer Science Undergraduate\nLocation: Western Province, Sri Lanka Email: ashinsa.rivindu@gmail.com Phone: +94 752 691 645\nWebsite: rivindu-ashinsa.github.io LinkedIn: rivindu-ashinsa GitHub: rivindu-ashinsa\nProfessional Summary\nComputer Science undergraduate with hands-on experience in data cleaning, analysis, and statistics. Proﬁcient\nin Python, SQL, and data reporting with proven ability to support data-driven decision-making. Experienced\nin processing and analyzing large datasets, creating actionable insights, and building data pipelines. Strong\nanalytical and problem-solving skills with expertise in statistical analysis and data interpret

### Embedding and VectorStoreDB

In [11]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Tuple, Any, Dict, Dict, List
from sklearn.metrics.pairwise import cosine_distances

In [10]:
sent_trans = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sent_trans.get_embedding_dimension()

# sent_trans.encode(["Hello"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3359.80it/s]


384

In [15]:
class EmbeddingManager:
    def __init__(self,model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model() 
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_embedding_dimension())

        except Exception as e:
            print(e)
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            print("Model not loaded.")
            self._load_model()
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated with shape:", embeddings.shape)
        return embeddings

In [16]:
## Initialize the Embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6353.98it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [ ]:
class VectorStore:
    def __init__(self, Collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = Collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        pass